
# Notebook 5: Data Validation

## Objective
Learn how to validate data using Python and Pandas to ensure
that a dataset follows defined quality rules and business constraints.

## Learning Outcomes
- Understand data validation.
- Perform range and type validation.
- Validate formats and categories.
- Detect missing required values.
- Apply business rules and referential integrity checks.
- Validate dates and data constraints.
- Create reusable validation rules using Python.


In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("Libraries imported successfully!")

Pandas version: 3.0.3
Libraries imported successfully!


In [2]:
data = {
    "Employee_ID": [101, 102, 103, 104, 105, 106],
    "Name": ["Ravi", "Priya", "Arun", "Sneha", None, "Kiran"],
    "Age": [25, 30, -5, 40, 28, 150],
    "Salary": [30000, 45000, 50000, -1000, 40000, 60000],
    "Percentage": [75, 88, 105, 65, 90, -10],
    "Gender": ["Male", "Female", "Male", "Other", "Unknown", "Female"],
    "Email": [
        "ravi@example.com",
        "priya@example.com",
        "arunexample.com",
        "sneha@example.com",
        "unknown@example.com",
        "kiran@example.com"
    ],
    "Date_of_Birth": [
        "2001-01-10",
        "1996-05-20",
        "2010-04-15",
        "1986-09-12",
        "1998-02-18",
        "1976-12-01"
    ],
    "Joining_Date": [
        "2024-01-10",
        "2022-05-15",
        "2023-03-01",
        "1980-01-01",
        "2025-02-20",
        "2020-12-01"
    ],
    "Department_ID": [1, 2, 3, 2, 1, 5]
}

df_validation = pd.DataFrame(data)

display(df_validation)

,Employee_ID,Name,Age,Salary,Percentage,Gender,Email,Date_of_Birth,Joining_Date,Department_ID
0,101,Ravi,25,30000,75,Male,ravi@example.com,2001-01-10,2024-01-10,1
1,102,Priya,30,45000,88,Female,priya@example.com,1996-05-20,2022-05-15,2
2,103,Arun,-5,50000,105,Male,arunexample.com,2010-04-15,2023-03-01,3
3,104,Sneha,40,-1000,65,Other,sneha@example.com,1986-09-12,1980-01-01,2
4,105,NaN,28,40000,90,Unknown,unknown@example.com,1998-02-18,2025-02-20,1
5,106,Kiran,150,60000,-10,Female,kiran@example.com,1976-12-01,2020-12-01,5



## 1. What Is Data Validation?

Data validation is the process of checking whether data follows
defined rules, expected formats, acceptable ranges, and business
requirements.

Validation helps identify incorrect, inconsistent, incomplete,
or logically impossible values before data analysis or machine
learning.

Examples:
- Age must be within an accepted range.
- Salary cannot be negative.
- Percentage must be between 0 and 100.
- Required fields cannot be empty.
- Dates must follow an accepted format.
- Foreign keys must refer to existing records.


In [3]:
# Age should be between 0 and 120.
invalid_age = df_validation[
    df_validation["Age"].notna()
    & ~df_validation["Age"].between(0, 120)
]

# Salary cannot be negative.
invalid_salary = df_validation[
    df_validation["Salary"].notna()
    & (df_validation["Salary"] < 0)
]

# Percentage must be between 0 and 100.
invalid_percentage = df_validation[
    df_validation["Percentage"].notna()
    & ~df_validation["Percentage"].between(0, 100)
]

print("Invalid ages:")
display(invalid_age[["Employee_ID", "Age"]])

print("Invalid salaries:")
display(invalid_salary[["Employee_ID", "Salary"]])

print("Invalid percentages:")
display(invalid_percentage[["Employee_ID", "Percentage"]])

Invalid ages:


,Employee_ID,Age
2,103,-5
5,106,150


Invalid salaries:


,Employee_ID,Salary
3,104,-1000


Invalid percentages:


,Employee_ID,Percentage
2,103,105
5,106,-10


In [4]:
print("Current data types:")
print(df_validation.dtypes)

print("\nAge is numeric:",
      pd.api.types.is_numeric_dtype(df_validation["Age"]))

print("Salary is numeric:",
      pd.api.types.is_numeric_dtype(df_validation["Salary"]))

Current data types:
Employee_ID      int64
Name               str
Age              int64
Salary           int64
Percentage       int64
Gender             str
Email              str
Date_of_Birth      str
Joining_Date       str
Department_ID    int64
dtype: object

Age is numeric: True
Salary is numeric: True


In [5]:
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

invalid_email = df_validation[
    df_validation["Email"].isna()
    | ~df_validation["Email"].astype("string").str.match(
        email_pattern, na=False
    )
]

print("Invalid email formats:")
display(invalid_email[["Employee_ID", "Email"]])

Invalid email formats:


,Employee_ID,Email
2,103,arunexample.com


In [6]:
allowed_genders = {"Male", "Female", "Other"}

invalid_gender = df_validation[
    df_validation["Gender"].isna()
    | ~df_validation["Gender"].isin(allowed_genders)
]

print("Invalid category values:")
display(invalid_gender[["Employee_ID", "Gender"]])

Invalid category values:


,Employee_ID,Gender
4,105,Unknown


In [7]:
required_columns = ["Employee_ID", "Name", "Age", "Salary"]

null_summary = df_validation[required_columns].isnull().sum()

print("Missing values in required columns:")
display(null_summary)

invalid_required_rows = df_validation[
    df_validation[required_columns].isnull().any(axis=1)
]

print("Rows missing required fields:")
display(invalid_required_rows)

Missing values in required columns:


Employee_ID    0
Name           1
Age            0
Salary         0
dtype: int64

Rows missing required fields:


,Employee_ID,Name,Age,Salary,Percentage,Gender,Email,Date_of_Birth,Joining_Date,Department_ID
4,105,NaN,28,40000,90,Unknown,unknown@example.com,1998-02-18,2025-02-20,1


In [8]:
# Business rule: an employee must be at least 18 years old.
underage_employees = df_validation[
    df_validation["Age"].notna()
    & (df_validation["Age"] < 18)
]

print("Employees below the minimum working age:")
display(underage_employees[["Employee_ID", "Age"]])

Employees below the minimum working age:


,Employee_ID,Age
2,103,-5


In [9]:
df_validation["Date_of_Birth"] = pd.to_datetime(
    df_validation["Date_of_Birth"],
    errors="coerce"
)

df_validation["Joining_Date"] = pd.to_datetime(
    df_validation["Joining_Date"],
    errors="coerce"
)

invalid_dates = df_validation[
    df_validation["Date_of_Birth"].isna()
    | df_validation["Joining_Date"].isna()
    | (
        df_validation["Date_of_Birth"].notna()
        & df_validation["Joining_Date"].notna()
        & (
            df_validation["Joining_Date"]
            < df_validation["Date_of_Birth"]
        )
    )
]

print("Rows with invalid or inconsistent dates:")
display(
    invalid_dates[
        ["Employee_ID", "Date_of_Birth", "Joining_Date"]
    ]
)

Rows with invalid or inconsistent dates:


,Employee_ID,Date_of_Birth,Joining_Date
3,104,1986-09-12,1980-01-01


In [10]:
valid_department_ids = {1, 2, 3, 4}

invalid_departments = df_validation[
    df_validation["Department_ID"].isna()
    | ~df_validation["Department_ID"].isin(valid_department_ids)
]

print("Invalid department references:")
display(invalid_departments[["Employee_ID", "Department_ID"]])

Invalid department references:


,Employee_ID,Department_ID
5,106,5


In [11]:
# Employee_ID must be unique and non-null.
duplicate_employee_ids = df_validation[
    df_validation["Employee_ID"].duplicated(keep=False)
]

missing_employee_ids = df_validation[
    df_validation["Employee_ID"].isna()
]

print("Duplicate employee IDs:")
display(duplicate_employee_ids)

print("Missing employee IDs:")
display(missing_employee_ids)

Duplicate employee IDs:


,Employee_ID,Name,Age,Salary,Percentage,Gender,Email,Date_of_Birth,Joining_Date,Department_ID


Missing employee IDs:


,Employee_ID,Name,Age,Salary,Percentage,Gender,Email,Date_of_Birth,Joining_Date,Department_ID


In [12]:
validation_report = pd.DataFrame({
    "Rule": [
        "Age range (0–120)",
        "Non-negative salary",
        "Percentage range (0–100)",
        "Valid email format",
        "Accepted gender category",
        "Required fields present",
        "Date consistency",
        "Valid department reference",
        "Unique employee ID"
    ],
    "Invalid_Count": [
        len(invalid_age),
        len(invalid_salary),
        len(invalid_percentage),
        len(invalid_email),
        len(invalid_gender),
        len(invalid_required_rows),
        len(invalid_dates),
        len(invalid_departments),
        len(duplicate_employee_ids)
    ]
})

validation_report["Status"] = np.where(
    validation_report["Invalid_Count"] == 0,
    "PASS",
    "REVIEW"
)

display(validation_report)

,Rule,Invalid_Count,Status
0,Age range (0–120),2,REVIEW
1,Non-negative salary,1,REVIEW
2,Percentage range (0–100),2,REVIEW
3,Valid email format,1,REVIEW
4,Accepted gender category,1,REVIEW
5,Required fields present,1,REVIEW
6,Date consistency,1,REVIEW
7,Valid department reference,1,REVIEW
8,Unique employee ID,0,PASS



## Conclusion

In this notebook, I learned how to validate data using Python
and Pandas.

I implemented range, type, format, category, null, business-rule,
referential, date, and constraint validation.

I also created a validation report to summarize records that
require investigation.

Data validation improves data quality and helps prevent incorrect
values from entering analysis or machine learning pipelines.
Validation rules should be documented and adapted to the dataset's
meaning and business requirements.
